In [2]:
import pandas as pd
import numpy as np
import joblib

grid = pd.read_parquet('../data/processed/daily_sales.parquet')

LAG_DAYS     = [1, 2, 3, 7, 14, 21, 28, 35, 42, 56]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
DOW_LAG_WKS  = [1, 2, 4, 8, 12]

def build_features(d, series):
    hist = series[series.index < d]
    row  = {
        'dow'           : d.dayofweek,
        'month'         : d.month,
        'dom'           : d.day,
        'quarter'       : d.quarter,
        'week'          : int(d.isocalendar().week),
        'is_weekend'    : int(d.dayofweek >= 5),
        'is_tet'        : int((d.month==1 and d.day>=15) or
                              (d.month==2 and d.day<=15)),
        'is_month_end'  : int(d.day >= 25),
        'is_month_start': int(d.day <= 5),
    }
    for lag in LAG_DAYS:
        row[f'lag_{lag}'] = float(
            series.get(d - pd.Timedelta(days=lag), 0.0))

    for w in ROLL_WINDOWS:
        win = hist.iloc[-w:] if len(hist) >= w else hist
        row[f'rmean_{w}'] = float(win.mean())
        row[f'rstd_{w}']  = float(win.std()) if len(win) > 1 else 0.
        row[f'rmax_{w}']  = float(win.max())
        row[f'rpos_{w}']  = float((win > 0).mean())

    same_dow = hist[hist.index.dayofweek == d.dayofweek]
    for wk in DOW_LAG_WKS:
        row[f'dlag_{wk}w'] = float(same_dow.iloc[-wk]) \
                              if len(same_dow) >= wk else 0.

    t28 = hist.iloc[-28:]
    row['trend_28'] = float(
        np.polyfit(np.arange(len(t28)), t28.values.astype(float), 1)[0]
    ) if len(t28) > 2 else 0.

    row['zero_frac_28'] = float((t28 == 0).mean()) if len(t28) > 0 else 1.

    ly     = d - pd.DateOffset(years=1)
    ly_win = hist[(hist.index >= ly - pd.Timedelta(days=14)) &
                  (hist.index <= ly + pd.Timedelta(days=14))]
    row['ly_mean'] = float(ly_win.mean()) if len(ly_win) > 0 else 0.

    return row

# Test on one SKU
s    = grid['SKU-09760']
test = build_features(pd.Timestamp('2025-08-01'), s)
print(f"Feature count : {len(test)}")
print(pd.Series(test).to_string())

print("Saved → ../models/build_features.pkl")

Feature count : 47
dow                  4.000000
month                8.000000
dom                  1.000000
quarter              3.000000
week                31.000000
is_weekend           0.000000
is_tet               0.000000
is_month_end         0.000000
is_month_start       1.000000
lag_1              154.000000
lag_2              228.000000
lag_3              600.000000
lag_7              100.000000
lag_14              10.000000
lag_21             100.000000
lag_28             130.000000
lag_35             160.000000
lag_42             128.000000
lag_56              30.000000
rmean_7            189.714286
rstd_7             202.169660
rmax_7             600.000000
rpos_7               0.857143
rmean_14           100.571429
rstd_14            165.883052
rmax_14            600.000000
rpos_14              0.714286
rmean_28           100.714286
rstd_28            135.468695
rmax_28            600.000000
rpos_28              0.750000
rmean_56           111.875000
rstd_56            22